# 02. Exploratory Data Analysis

Tổng hợp phân phối nhãn, độ dài văn bản và độ dài subword PhoBERT.

## 1. Setup

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown, Image

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd

print("Project root:", ROOT)

TABLES_DIR = ROOT / "outputs" / "tables"
FIGURES_DIR = ROOT / "outputs" / "figures"
REPORTS_DIR = ROOT / "outputs" / "reports"
PROCESSED_DIR = ROOT / "data" / "processed"

## 2. Kiểm tra file EDA bắt buộc

In [ ]:
required_files = {
    "all_standardized": PROCESSED_DIR / "all_standardized.csv",
    "basic_summary": TABLES_DIR / "eda_basic_summary.csv",
    "sentiment_distribution": TABLES_DIR / "eda_label_distribution_sentiment.csv",
    "topic_distribution": TABLES_DIR / "eda_label_distribution_topic.csv",
    "text_length_percentiles": TABLES_DIR / "text_length_percentiles.csv",
    "subword_percentiles": TABLES_DIR / "phobert_subword_length_percentiles.csv",
    "max_length_recommendation": TABLES_DIR / "max_length_recommendation.csv",
    "sentiment_figure": FIGURES_DIR / "sentiment_distribution.png",
    "topic_figure": FIGURES_DIR / "topic_distribution.png",
    "subword_figure": FIGURES_DIR / "phobert_subword_count_distribution.png",
}

check_df = pd.DataFrame(
    [{"name": name, "path": str(path), "exists": path.exists()} for name, path in required_files.items()]
)

display(check_df)

missing = check_df.loc[~check_df["exists"], "name"].tolist()
if missing:
    raise FileNotFoundError(f"Missing required EDA files: {missing}")

print("All required Stage 2 files exist.")

## 3. Basic summary

Nội dung: kiểm tra số mẫu, text rỗng, duplicate và số lớp của từng task.

In [ ]:
basic_summary = pd.read_csv(TABLES_DIR / "eda_basic_summary.csv")
display(basic_summary)

## 4. Sentiment label distribution

Nội dung: kiểm tra mức mất cân bằng giữa `negative`, `neutral`, `positive`.

In [ ]:
sentiment_dist = pd.read_csv(TABLES_DIR / "eda_label_distribution_sentiment.csv")
display(sentiment_dist)
display(Image(filename=str(FIGURES_DIR / "sentiment_distribution.png")))

### Nhận xét sentiment

Cần chú ý lớp `neutral`. Nếu tỷ lệ lớp này rất nhỏ, Accuracy sẽ không đủ đáng tin. Khi đó, Macro-F1 và per-class F1 phải được dùng làm metric chính.

In [ ]:
sentiment_pivot = sentiment_dist.pivot(index="split", columns="label_name", values="percent")
display(sentiment_pivot)

if "neutral" in sentiment_pivot.columns:
    neutral_rates = sentiment_pivot["neutral"]
    display(Markdown(f"Neutral percent range: **{neutral_rates.min():.2f}% - {neutral_rates.max():.2f}%**"))

## 5. Topic label distribution

Nội dung: kiểm tra mức mất cân bằng giữa `lecturer`, `training_program`, `facility`, `others`.

In [ ]:
topic_dist = pd.read_csv(TABLES_DIR / "eda_label_distribution_topic.csv")
display(topic_dist)
display(Image(filename=str(FIGURES_DIR / "topic_distribution.png")))

### Nhận xét topic

Nếu `lecturer` chiếm đa số, model có thể đạt Accuracy cao dù dự đoán kém các lớp nhỏ như `facility` và `others`. Vì vậy topic classification phải ưu tiên Macro-F1.

In [ ]:
topic_pivot = topic_dist.pivot(index="split", columns="label_name", values="percent")
display(topic_pivot)

if "lecturer" in topic_pivot.columns:
    lecturer_rates = topic_pivot["lecturer"]
    display(Markdown(f"Lecturer percent range: **{lecturer_rates.min():.2f}% - {lecturer_rates.max():.2f}%**"))

## 6. Text length analysis

`whitespace_word_count` chỉ là độ dài theo khoảng trắng, không phải word segmentation tiếng Việt chuẩn. Tuy nhiên, nó vẫn hữu ích để hiểu độ dài phản hồi ở mức sơ bộ.

In [ ]:
word_percentiles = pd.read_csv(TABLES_DIR / "text_length_percentiles.csv")
display(word_percentiles)

## 7. PhoBERT subword length analysis

Nội dung: ước lượng độ dài input khi dùng tokenizer PhoBERT.

In [ ]:
subword_percentiles = pd.read_csv(TABLES_DIR / "phobert_subword_length_percentiles.csv")
display(subword_percentiles)
display(Image(filename=str(FIGURES_DIR / "phobert_subword_count_distribution.png")))

## 8. max_length recommendation

`max_length` nên được chọn theo coverage, không chọn cảm tính.

In [ ]:
max_length_df = pd.read_csv(TABLES_DIR / "max_length_recommendation.csv")
display(max_length_df)

# Gợi ý tự động: chọn candidate nhỏ nhất có coverage >= 99.9%
candidate = max_length_df.loc[max_length_df["coverage_percent"] >= 99.9].head(1)

if len(candidate) > 0:
    recommended = int(candidate.iloc[0]["max_length_candidate"])
else:
    recommended = int(max_length_df.sort_values("coverage_percent", ascending=False).iloc[0]["max_length_candidate"])

display(Markdown(f"Recommended max_length from >=99.9% coverage rule: **{recommended}**"))

# Với robustness evaluation, ta vẫn có thể chọn 128 để có buffer cho noisy text.
if recommended < 128:
    display(Markdown("For this project, choose **max_length = 128** to keep extra buffer for noisy text evaluation."))
else:
    display(Markdown(f"For this project, choose **max_length = {recommended}**."))

## 9. Kết luận Stage 2

Ghi nhận cho các giai đoạn sau:

```text
- Dataset không có text rỗng.
- Sentiment mất cân bằng, đặc biệt lớp neutral nhỏ.
- Topic mất cân bằng mạnh, lecturer chiếm đa số.
- Macro-F1 là metric chính cho cả hai task.
- Cần xem per-class F1 và confusion matrix, không chỉ Accuracy.
- Văn bản phần lớn ngắn.
- max_length=128 là lựa chọn an toàn cho PhoBERT vì gần như phủ toàn bộ dữ liệu sạch và có buffer cho noisy text.
```

Quyết định kỹ thuật:

```text
configs/phobert_sentiment.yaml → training.max_length = 128
configs/phobert_topic.yaml     → training.max_length = 128
```

Giai đoạn tiếp theo:

```text
Stage 3 — Baseline Models
```